# Table 2 – Accuracy & ARI Analysis

Reads `.Rds` simulation outputs from the **enhanced** pipeline and produces
the LaTeX table (Accuracy + ARI, by censoring / k / γ, scenario = `baseline`).

**Expected directory layout** (mirrors `enhanced_simulation_main.R`):
```
output/
  tab2/
    baseline/
      sim_seed*_c*_k*_gammapar*_frailty*_censor*.Rds
```

In [15]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import rdata      
from pathlib import Path
from sklearn.metrics import adjusted_rand_score

# ── Configuration ──────────────────────────────────────────────────────────
BASE_DIR  = Path("output/tab2")   # root output folder from enhanced_simulation_main.R
SCENARIO  = "baseline"            # which sub-folder / scenario to analyse
N_CLUSTERS_EXPECTED = 3           # keep only runs where n_components == 3

folder = BASE_DIR / SCENARIO
rds_files = sorted(folder.glob("*.Rds"))
print(f"Found {len(rds_files)} .Rds files in '{folder}'")

Found 2800 .Rds files in 'output/tab2/baseline'


In [16]:
# ── Helper: greedy relabelling (predicted → true) ──────────────────────────
def simple_relabel(true_labels, pred_labels):
    """Map predicted cluster ids to true ids by maximum overlap (greedy)."""
    true_labels = pd.Series(true_labels)
    pred_labels = pd.Series(pred_labels)

    mapping = {}
    available_true = set(true_labels.unique())

    for p in pred_labels.unique():
        overlaps = {
            t: ((true_labels == t) & (pred_labels == p)).sum()
            for t in available_true
        }
        best = max(overlaps, key=overlaps.get)
        mapping[p] = best
        available_true.discard(best)

    return pred_labels.map(mapping)

In [17]:
# ── Load & process all .Rds files ─────────────────────────────────────────
#
# Each file is an R list saved with saveRDS().

# Expected fields (from enhanced_simulation_main.R):
#   seed, scenario, c, k, gammapar, n_components, clusters (Nx2 matrix),
#   dgp_frailty, dgp_baseline, fit_frailty, fit_baseline,
#   separation, balance, censoring_rate
#
# The censoring type is encoded in the filename:  _censoradministrative_
# or _censornormal_

results = {}   # (censor_name, k, gamma) → {"acc": [], "ari": []}
skipped = 0

for file in rds_files:
    fname = file.name

    # ── Determine censoring from filename ──────────────────────────────
    if "censoradministrative" in fname:
        censor_name = "Administrative"
    elif "censornormal" in fname:
        censor_name = "Normal"
    else:
        skipped += 1
        continue

    # ── Read the Rds file ──────────────────────────────────────────────
    try:
        obj = rdata.read_rds(file)
        obj = {str(k): v for k, v in obj.items()}

    except Exception as e:
        print(f"  [WARNING] Could not read {fname}: {e}")
        skipped += 1
        continue

    # ── Filter: only runs with the expected number of components ───────
    n_comp = obj.get("n_components", None)
    if n_comp is None or int(np.asarray(n_comp).item()) != N_CLUSTERS_EXPECTED:
        skipped += 1
        continue

    # ── Extract k and gamma ────────────────────────────────────────────
    k     = int(np.asarray(obj["k"]).item())
    gamma = float(np.asarray(obj["gammapar"]).item())

    # ── Compute accuracy & ARI ─────────────────────────────────────────
    clusters_arr = np.asarray(obj["clusters"])

    if clusters_arr.ndim != 2 or clusters_arr.shape[1] != 2:
        print(f"[WARNING] Bad clusters shape in {fname}: {clusters_arr.shape}")
        skipped += 1
        continue

    true_labels = clusters_arr[:, 0]
    pred_labels = clusters_arr[:, 1]

    # relabel + numpy arrays
    aligned = simple_relabel(true_labels, pred_labels)

    true_labels = np.asarray(true_labels)
    aligned     = np.asarray(aligned)

    accuracy = (true_labels == aligned).mean()
    ari      = adjusted_rand_score(true_labels, aligned)

    # ── KEY FIX ───────────
    key = (censor_name, k, gamma)

    if key not in results:
        results[key] = {"acc": [], "ari": []}

    results[key]["acc"].append(accuracy)
    results[key]["ari"].append(ari)

print(f"Processed {len(rds_files) - skipped} files  |  skipped {skipped}")

/opt/homebrew/lib/python3.11/site-packages/rdata/conversion/_conversion.py:900: UserWarning: Missing constructor for R class "table". The underlying R object is returned instead.
  warnings.warn(


Processed 2779 files  |  skipped 21


In [18]:
# ── Build summary DataFrame ────────────────────────────────────────────────
rows = []
for (censor, k, gamma), vals in results.items():
    acc = np.array(vals["acc"])
    ari = np.array(vals["ari"])
    rows.append({
        "Censoring" : censor,
        "k"         : k,
        "gamma"     : gamma,
        "Acc_mean"  : np.round(acc.mean(), 3),
        "Acc_median": np.round(np.median(acc), 3),
        "Acc_sd"    : np.round(acc.std(), 3),
        "ARI_mean"  : np.round(ari.mean(), 3),
        "ARI_median": np.round(np.median(ari), 3),
        "ARI_sd"    : np.round(ari.std(), 3),
        "n_reps"    : len(acc),
    })

# Sort: Administrative first, then Normal; within each: k asc, gamma asc
censor_order = {"Administrative": 0, "Normal": 1}
summary_df = (
    pd.DataFrame(rows)
    .assign(censor_ord=lambda df: df["Censoring"].map(censor_order))
    .sort_values(["censor_ord", "k", "gamma"])
    .drop(columns="censor_ord")
    .reset_index(drop=True)
)

print(summary_df.to_string())

         Censoring   k     gamma  Acc_mean  Acc_median  Acc_sd  ARI_mean  ARI_median  ARI_sd  n_reps
0   Administrative  20  0.000001     0.971       1.000   0.089     0.942       1.000   0.161     100
1   Administrative  20  0.000100     0.971       1.000   0.089     0.942       1.000   0.161     100
2   Administrative  20  0.001000     0.975       1.000   0.081     0.950       1.000   0.147     100
3   Administrative  20  0.010000     0.957       1.000   0.082     0.908       1.000   0.172     100
4   Administrative  20  0.100000     0.950       1.000   0.083     0.888       1.000   0.179     100
5   Administrative  20  0.200000     0.930       1.000   0.115     0.860       1.000   0.197      99
6   Administrative  20  0.400000     0.891       0.928   0.144     0.788       0.819   0.242      95
7   Administrative  50  0.000001     1.000       1.000   0.001     0.999       1.000   0.003     100
8   Administrative  50  0.000100     1.000       1.000   0.001     0.999       1.000   0.00

In [19]:
# ── Identify best rows (bold in LaTeX) ────────────────────────────────────
#
# «Best» = highest Acc_mean  (ties broken by lowest Acc_sd).
# Applied independently within each (Censoring, k) group.
# A row is bold when ALL SIX values equal those of the best row.

def find_bold_rows(group):
    """Return boolean Series: True for row(s) to bold in group."""
    # Best accuracy: max mean, then min sd
    max_acc_mean = group["Acc_mean"].max()
    candidates   = group[group["Acc_mean"] == max_acc_mean]
    min_acc_sd   = candidates["Acc_sd"].min()
    best_mask    = (group["Acc_mean"] == max_acc_mean) & (group["Acc_sd"] == min_acc_sd)
    return best_mask

bold_mask = pd.Series(False, index=summary_df.index)
for (censor, k), grp in summary_df.groupby(["Censoring", "k"], sort=False):
    bold_mask.loc[grp.index] = find_bold_rows(grp)

summary_df["bold"] = bold_mask
print("Bold rows:")
print(summary_df[summary_df["bold"]][["Censoring","k","gamma","Acc_mean","ARI_mean"]])

Bold rows:
         Censoring   k     gamma  Acc_mean  ARI_mean
2   Administrative  20  0.001000     0.975     0.950
7   Administrative  50  0.000001     1.000     0.999
8   Administrative  50  0.000100     1.000     0.999
9   Administrative  50  0.001000     1.000     0.999
10  Administrative  50  0.010000     1.000     0.999
19          Normal  20  0.200000     0.922     0.831
24          Normal  50  0.010000     0.998     0.994


# LaTex table

In [21]:
# ── LaTeX table generator ──────────────────────────────────────────────────

GAMMA_ORDER = [0, 1e-6, 1e-4, 1e-3, 1e-2, 0.1, 0.2, 0.4]  # display order

def fmt_gamma(g):
    """Format a gamma value as LaTeX scientific / decimal."""
    if g == 0:
        return r"$0$"
    elif g < 0.01:
        e = int(round(np.log10(g)))
        return rf"$10^{{{e}}}$"
    else:
        return f"${g:g}$"

def fmt_val(v, bold=False):
    s = f"{v:.3f}"
    return rf"\textbf{{{s}}}" if bold else s


def build_latex_table(df):
    lines = []
    lines.append(r"""\begin{table}[!htbp]
\centering
\footnotesize
\begin{tabular}{lll ccc ccc}
\toprule
\textbf{Censoring} & $k$ & $\gamma$
    & \multicolumn{3}{c}{\textbf{Accuracy}}
    & \multicolumn{3}{c}{\textbf{ARI}} \\\\
\cmidrule(lr){4-6} \cmidrule(lr){7-9}
 & & & Mean & Median & SD & Mean & Median & SD \\\\
\toprule""")

    censor_groups = df.groupby("Censoring", sort=False)
    censors = df["Censoring"].unique()  # preserves sorted order

    for ci, censor in enumerate(censors):
        cdf = df[df["Censoring"] == censor]
        ks  = sorted(cdf["k"].unique())
        n_rows_censor = len(cdf)

        censor_label = r"(i) Administrative" if censor == "Administrative" else r"(ii) Normal"

        if ci > 0:
            lines.append(r"\midrule")

        first_censor_row = True
        for ki, k in enumerate(ks):
            kdf = cdf[cdf["k"] == k].copy()
            # Enforce gamma display order
            kdf["_gord"] = kdf["gamma"].map(
                {g: i for i, g in enumerate(GAMMA_ORDER)}
            ).fillna(999)
            kdf = kdf.sort_values("_gord").drop(columns="_gord")

            n_rows_k = len(kdf)

            if ki > 0:
                lines.append(r"  \cmidrule(lr){2-9}")

            for ri, (_, row) in enumerate(kdf.iterrows()):
                b = row["bold"]

                # Column 1: censoring label (multirow, first row of whole block)
                if first_censor_row and ri == 0:
                    col1 = rf"\multirow{{{n_rows_censor}}}{{*}}{{{censor_label}}}"
                else:
                    col1 = ""

                # Column 2: k (multirow, first row of each k block)
                if ri == 0:
                    col2 = rf"  & \multirow{{{n_rows_k}}}{{*}}{{{k}}}"
                else:
                    col2 = "  &"

                gamma_str = fmt_gamma(row["gamma"])

                data_cols = " & ".join([
                    fmt_val(row["Acc_mean"],   b),
                    fmt_val(row["Acc_median"], b),
                    fmt_val(row["Acc_sd"],     b),
                    fmt_val(row["ARI_mean"],   b),
                    fmt_val(row["ARI_median"], b),
                    fmt_val(row["ARI_sd"],     b),
                ])

                line = f"  {col1}{col2} & {gamma_str} & {data_cols}\\\\"
                lines.append(line)
                first_censor_row = False

    lines.append(r"""\bottomrule
\end{tabular}
\caption{\small Empirical means, medians, and standard deviations of \textit{accuracy} and \textit{ARI} over $B=100$ replications, evaluated under two censoring mechanisms and across varying values of $k$ and $\gamma$, setting $C=3$. The best estimates in each setting are highlighted in bold.}
\label{tab:AccuracyandARI}
\end{table}""")

    return "\n".join(lines)


latex_str = build_latex_table(summary_df)
print(latex_str)

\begin{table}[!htbp]
\centering
\footnotesize
\begin{tabular}{lll ccc ccc}
\toprule
\textbf{Censoring} & $k$ & $\gamma$
    & \multicolumn{3}{c}{\textbf{Accuracy}}
    & \multicolumn{3}{c}{\textbf{ARI}} \\\\
\cmidrule(lr){4-6} \cmidrule(lr){7-9}
 & & & Mean & Median & SD & Mean & Median & SD \\\\
\toprule
  \multirow{14}{*}{(i) Administrative}  & \multirow{7}{*}{20} & $10^{-6}$ & 0.971 & 1.000 & 0.089 & 0.942 & 1.000 & 0.161\\
    & & $10^{-4}$ & 0.971 & 1.000 & 0.089 & 0.942 & 1.000 & 0.161\\
    & & $10^{-3}$ & \textbf{0.975} & \textbf{1.000} & \textbf{0.081} & \textbf{0.950} & \textbf{1.000} & \textbf{0.147}\\
    & & $0.01$ & 0.957 & 1.000 & 0.082 & 0.908 & 1.000 & 0.172\\
    & & $0.1$ & 0.950 & 1.000 & 0.083 & 0.888 & 1.000 & 0.179\\
    & & $0.2$ & 0.930 & 1.000 & 0.115 & 0.860 & 1.000 & 0.197\\
    & & $0.4$ & 0.891 & 0.928 & 0.144 & 0.788 & 0.819 & 0.242\\
  \cmidrule(lr){2-9}
    & \multirow{7}{*}{50} & $10^{-6}$ & \textbf{1.000} & \textbf{1.000} & \textbf{0.001} & \textbf{0.

In [ ]:
# ── Save LaTeX table to file ───────────────────────────────────────────────
# out_path = Path("Tab2_AccuracyARI.tex")
# out_path.write_text(latex_str)
# print(f"LaTeX table written to {out_path.resolve()}")

LaTeX table written to /Users/alessandragni/Documents/DATA/POLITECNICO/PHD/CODE_REPO/HazClust/simulations/Tab2_AccuracyARI.tex
